# Short-Term Memory

In [1]:
from agents import Agent, Runner, SQLiteSession, trace
from setup import bedrock_tool

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


In [2]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant comparing how healthy different foods are.
    If you answer, give a list of how healthy the foods are with a score from 1 to 10. Order by: healtiest food comes first.

    Example:
    Q: Compare X and Y
    A: X is healtier as Y.
    1) X: 8/10 - Very healthy but high in fructose
    2) Y: 3/10 - High in sugar and fat
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
)

## No Memory

In [3]:
with trace("Compare without Memory") as t:
    result = await Runner.run(nutrition_agent, "Which is healthier, bananas or lollipop?")

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")


Output: Certainly! Here's a comparison of the healthiness of bananas and lollipops, with scores from 1 to 10:

1) Bananas: 9/10 - Bananas are very healthy. They are rich in essential nutrients such as potassium, vitamin B6, vitamin C, and dietary fiber. They also provide natural sugars and are a good source of energy.

2) Lollipops: 1/10 - Lollipops are high in added sugars and provide very little to no nutritional value. They can contribute to tooth decay and provide empty calories with no health benefits.

So, bananas are significantly healthier than lollipops.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_5db1a1c3b6e94b58812e7ca60ad163e0


In [4]:
with trace("Add Apple - without Memory") as t:
    result = await Runner.run(nutrition_agent, "Add apples to the comparison")

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: Certainly! Let's compare apples to a few other foods and rate their healthiness on a scale of 1 to 10.

1) **Apples: 9/10**
   - **Pros:** High in fiber, vitamin C, and antioxidants. Low in calories.
   - **Cons:** Contains natural sugars.

2) **Spinach: 10/10**
   - **Pros:** Rich in vitamins A, C, and K, as well as iron, calcium, and antioxidants. Very low in calories.
   - **Cons:** None significant for general consumption.

3) **Blueberries: 8/10**
   - **Pros:** High in antioxidants, vitamin C, and fiber. Low in calories.
   - **Cons:** Contains natural sugars.

4) **Salmon: 8/10**
   - **Pros:** Excellent source of omega-3 fatty acids, high-quality protein, and various vitamins and minerals.
   - **Cons:** Higher in calories compared to fruits; concerns with contaminants if not sustainably sourced.

5) **Whole Grains (e.g., brown rice, quinoa): 7/10**
   - **Pros:** High in fiber, B vitamins, and minerals. Can help maintain digestive health.
   - **Cons:** May contain ant

## Short Term Memory

In [5]:
session = SQLiteSession("conversation_history")

In [6]:
result = await Runner.run(
    nutrition_agent, "Which is healthier, bananas or lollipop?", session=session
)
print(result.final_output)

A: Bananas are healthier than lollipops.

1) Bananas: 9/10 - High in fiber, vitamins (especially vitamin C and B6), minerals (like potassium), and antioxidants. They are also low in fat and calories.

2) Lollipops: 1/10 - High in sugar and artificial additives, with no significant nutritional benefits. They provide empty calories and can contribute to dental problems.


In [7]:
with trace("Compare with Memory") as t:
    result = await Runner.run(
        nutrition_agent, "Add apples to the comparison", session=session
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: A: Bananas are healthier than lollipops, followed by apples.

1) Bananas: 9/10 - High in fiber, vitamins (especially vitamin C and B6), minerals (like potassium), and antioxidants. They are also low in fat and calories.

2) Apples: 8/10 - High in fiber, vitamin C, and various antioxidants. They are also low in calories and have been linked to several health benefits, such as improved heart health.

3) Lollipops: 1/10 - High in sugar and artificial additives, with no significant nutritional benefits. They provide empty calories and can contribute to dental problems.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_bc5881971b7844df89f3fbb92e49547d
